# Semantic Axis Exploration

This notebook explores semantic opposition axes in literary text using sentence-transformer embeddings.

A semantic axis is defined by two sets of anchor examples representing opposite poles, such as `light` and `dark`. Each pole is represented by a centroid in embedding space. The axis vector is computed as the difference between the two centroids, and each text segment is projected onto that axis.

This differs from the anchored sentiment scoring procedure used in `02_anchored_sentiment_analysis.ipynb`. In that notebook, the score is computed as the difference between mean similarity to positive and negative anchors. In this notebook, the score is computed by projecting each segment embedding onto a constructed semantic axis.

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

## Load Sentence Transformer model

In [ ]:
# Replace this with your actual model path
MODEL_PATH = Path("../models/tolkien_sentence_transformer_epoch_1")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Could not find model at {MODEL_PATH}. "
        "Update MODEL_PATH to your local sentence-transformer model directory."
    )

model = SentenceTransformer(str(MODEL_PATH))

## Preprocessing and segmentation

The sentiment-scoring section expects a dataframe named `segments_df` with at least the following columns:

- `segment_text`: the text segment to be scored;
- `chapter`: the chapter or section label aligned with the segment;
- `segment_type`: optional information about the segment type, such as prose, dialogue, verse, or unknown.

Different source formats require different preprocessing rules. Where possible, structured formats such as TEI/XML or HTML should be preferred because paragraph, chapter, and verse boundaries may already be marked. When only plain text is available, source-specific rules must be checked and documented.

The optional parser cells below illustrate possible preprocessing routes. The default workflow in this notebook uses a prepared plain-text file with explicit chapter markers.

The prepared plain-text parser below is used for the local LOTR file in this project. It assumes chapter markers in the form `###CHAPTER:` and a local paragraph-start convention based on leading spaces. These rules are source-specific.

### TEI/XML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

TEI_PATH = Path("data/source.xml")

def segments_from_tei(path: Path) -> pd.DataFrame:
    """
    Extract prose paragraphs and verse blocks from a TEI/XML file.

    This is a template parser. TEI structures vary, so tag names and
    attributes may need to be adapted for a specific corpus.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "xml")

    rows = []

    for div in soup.find_all("div"):
        chapter_head = div.find("head")
        chapter = chapter_head.get_text(" ", strip=True) if chapter_head else "Unknown"

        for element in div.find_all(["p", "lg"], recursive=True):
            if element.name == "p":
                segment_type = "prose"
                text = element.get_text(" ", strip=True)

            elif element.name == "lg":
                segment_type = "verse"
                lines = [line.get_text(" ", strip=True) for line in element.find_all("l")]
                text = " / ".join(lines) if lines else element.get_text(" ", strip=True)

            else:
                continue

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": chapter,
                        "segment_type": segment_type,
                    }
                )

    return pd.DataFrame(rows)

### HTML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

HTML_PATH = Path("data/source.html")

def segments_from_html(path: Path) -> pd.DataFrame:
    """
    Extract paragraphs from an HTML file.

    This works best for HTML/EPUB-derived texts where paragraphs are
    marked with <p> tags. Chapter detection may need to be adapted
    depending on the source.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    rows = []
    current_chapter = "Unknown"

    for element in soup.find_all(["h1", "h2", "h3", "p"]):
        if element.name in ["h1", "h2", "h3"]:
            current_chapter = element.get_text(" ", strip=True)

        elif element.name == "p":
            text = element.get_text(" ", strip=True)

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": current_chapter,
                        "segment_type": "prose",
                    }
                )

    return pd.DataFrame(rows)

## Prepared plain-text parser used in this notebook

In [ ]:
# Cell — Segment corpus

# -------------------------------
# Segmentation parameters
# -------------------------------
MIN_TOK_NARR, MAX_TOK_NARR = 80, 300
MIN_TOK_DIAL, MAX_TOK_DIAL = 60, 180
MAX_DIALOGUE_TURNS = 6

DIALOG_START_CHARS = ('"', "'", "“", "”", "‘", "’", "—", "–", "-", "―")

CORPUS_PATH = Path("data/LotR.txt")

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CORPUS_PATH}. "
        "The source text is not included in the public repository because it contains copyrighted text."
    )


def is_dialogue_first_line(line: str) -> bool:
    """Return True if a line appears to begin with dialogue punctuation."""
    return line.lstrip().startswith(DIALOG_START_CHARS)


def is_paragraph_lead(line: str) -> bool:
    """Return True if the line follows the local five-space paragraph convention."""
    return line.startswith("     ")


def tokenize_count(text: str) -> int:
    """Count word tokens, treating hyphenated words as single tokens."""
    return len(re.findall(r"\b\w+(?:-\w+)*\b", text))


def build_raw_paragraphs(lines: List[str]) -> Tuple[List[str], List[str], List[bool]]:
    """Build raw paragraphs with aligned chapter labels and dialogue flags."""
    paragraphs: List[str] = []
    para_chapters: List[str] = []
    para_is_dialogue: List[bool] = []

    current_chapter = "Unknown"
    buf_lines: List[str] = []
    buf_is_dialogue = False

    def flush_paragraph():
        nonlocal buf_lines, buf_is_dialogue

        if buf_lines:
            paragraphs.append(" ".join(ln.strip() for ln in buf_lines).strip())
            para_chapters.append(current_chapter)
            para_is_dialogue.append(buf_is_dialogue)

        buf_lines = []
        buf_is_dialogue = False

    for raw in lines:
        line = raw.rstrip("\r")

        if line.startswith("###CHAPTER:"):
            flush_paragraph()
            current_chapter = line.replace("###CHAPTER:", "").strip()
            continue

        if line.strip() == "":
            flush_paragraph()
            continue

        if is_paragraph_lead(line):
            flush_paragraph()
            buf_lines = [line]
            buf_is_dialogue = is_dialogue_first_line(line)
        else:
            if buf_lines:
                buf_lines.append(line)
            else:
                buf_lines = [line]
                buf_is_dialogue = is_dialogue_first_line(line)

    flush_paragraph()
    return paragraphs, para_chapters, para_is_dialogue


def merge_dialogue_aware(
    paragraphs: List[str],
    para_chapters: List[str],
    para_is_dialogue: List[bool],
) -> Tuple[List[str], List[str]]:
    """
    Merge raw paragraphs into dialogue-aware analysis segments.

    Chapter boundaries are preserved: segments are never merged across chapters.
    """
    processed_paragraphs: List[str] = []
    chapter_tags: List[str] = []

    seg_buf: List[str] = []
    seg_tokens = 0
    seg_is_dialogue = None
    seg_dialogue_turns = 0
    seg_chapter = None

    def seg_flush():
        nonlocal seg_buf, seg_tokens, seg_is_dialogue, seg_dialogue_turns, seg_chapter

        if seg_buf:
            processed_paragraphs.append(" ".join(seg_buf).strip())
            chapter_tags.append(seg_chapter)

        seg_buf = []
        seg_tokens = 0
        seg_is_dialogue = None
        seg_dialogue_turns = 0
        seg_chapter = None

    for paragraph, chapter, is_dialogue in zip(
        paragraphs,
        para_chapters,
        para_is_dialogue,
    ):
        paragraph_tokens = tokenize_count(paragraph)

        if paragraph_tokens == 0:
            continue

        if seg_buf and chapter != seg_chapter:
            seg_flush()

        if not seg_buf:
            seg_buf = [paragraph]
            seg_tokens = paragraph_tokens
            seg_is_dialogue = is_dialogue
            seg_dialogue_turns = 1 if is_dialogue else 0
            seg_chapter = chapter
            continue

        max_tokens = MAX_TOK_DIAL if seg_is_dialogue else MAX_TOK_NARR
        min_tokens = MIN_TOK_DIAL if seg_is_dialogue else MIN_TOK_NARR

        type_switch = is_dialogue != seg_is_dialogue
        exceeds_limits = (
            seg_tokens + paragraph_tokens > max_tokens
            or (seg_is_dialogue and seg_dialogue_turns >= MAX_DIALOGUE_TURNS)
        )

        if type_switch or exceeds_limits:
            if seg_tokens >= min_tokens:
                seg_flush()

                seg_buf = [paragraph]
                seg_tokens = paragraph_tokens
                seg_is_dialogue = is_dialogue
                seg_dialogue_turns = 1 if is_dialogue else 0
                seg_chapter = chapter
                continue

        seg_buf.append(paragraph)
        seg_tokens += paragraph_tokens

        if is_dialogue and seg_is_dialogue:
            seg_dialogue_turns += 1
        elif type_switch:
            seg_is_dialogue = is_dialogue
            seg_dialogue_turns = 1 if is_dialogue else 0

    seg_flush()
    return processed_paragraphs, chapter_tags


with open(CORPUS_PATH, "r", encoding="utf-8") as file:
    lines = file.read().split("\n")

raw_paragraphs, raw_chapter_tags, raw_is_dialogue = build_raw_paragraphs(lines)

processed_paragraphs, chapter_tags = merge_dialogue_aware(
    raw_paragraphs,
    raw_chapter_tags,
    raw_is_dialogue,
)

print(f"Raw paragraphs: {len(raw_paragraphs)}")
print(f"Processed segments: {len(processed_paragraphs)}")
print(f"Unique chapters: {len(set(chapter_tags))}")

In [ ]:
# Cell — Source-format diagnostics

chapter_marker_count = sum(
    line.startswith("###CHAPTER:")
    for line in lines
)

paragraph_lead_count = sum(
    is_paragraph_lead(line)
    for line in lines
)

blank_line_count = sum(
    line.strip() == ""
    for line in lines
)

source_diagnostics = {
    "chapter_markers": chapter_marker_count,
    "paragraph_lead_lines": paragraph_lead_count,
    "blank_lines": blank_line_count,
    "raw_paragraphs": len(raw_paragraphs),
    "processed_segments": len(processed_paragraphs),
}

source_diagnostics

## Standard segment dataframe

The segmentation step is standardised into a dataframe named `segments_df`. The later sentiment-scoring cells use this dataframe rather than depending on the specific preprocessing method that produced the segments.

At minimum, `segments_df` contains the segment text and aligned chapter label. A `segment_type` column is included so that future versions can distinguish narration, dialogue, verse, or mixed segments.

In the current plain-text parser, final merged segments are labelled as `mixed_or_unknown`. Future versions may propagate `narration`, `dialogue`, or `verse` labels into the final dataframe.

In [ ]:
# Cell — Build standard segments dataframe

segments_df = pd.DataFrame(
    {
        "segment_text": processed_paragraphs,
        "chapter": chapter_tags,
        "segment_type": "mixed_or_unknown",
    }
)

segments_df["token_count"] = segments_df["segment_text"].apply(tokenize_count)

segments_df.head()

In [ ]:
# Check segment counts by chapter
chapter_segment_counts = (
    segments_df["chapter"]
    .value_counts()
    .sort_index()
)

chapter_segment_counts.head()

In [ ]:
# Cell — Segmentation diagnostics

segment_lengths = [tokenize_count(segment) for segment in processed_paragraphs]

diagnostics = {
    "raw_paragraphs": len(raw_paragraphs),
    "processed_segments": len(processed_paragraphs),
    "unique_chapters": len(set(chapter_tags)),
    "min_segment_tokens": min(segment_lengths),
    "max_segment_tokens": max(segment_lengths),
    "mean_segment_tokens": np.mean(segment_lengths),
    "median_segment_tokens": np.median(segment_lengths),
}

diagnostics

## Semantic-axis construction

This section constructs a Light–Dark semantic axis from two sets of anchor sentences.

This differs from the anchored sentiment scoring used in notebook 02. Here, the model first computes a centroid for each semantic pole. The axis is then defined as the direction from the Dark centroid to the Light centroid, and each text segment is projected onto that direction.

## Define Light and Dark anchors

The current light/dark axis uses 13 light anchors and 13 dark anchors. The anchor texts are not included because they contain copyrighted material.

In [ ]:
AXIS_ANCHORS_PATH = Path("data/light_dark_anchors.csv")

if not AXIS_ANCHORS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {AXIS_ANCHORS_PATH}. "
        "Axis anchor sentences are not included in the public repository because they contain copyrighted text."
    )

axis_anchors_df = pd.read_csv(AXIS_ANCHORS_PATH, sep=";")

required_cols = {"sentence_text", "axis_label"}
missing = required_cols - set(axis_anchors_df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Present columns: {list(axis_anchors_df.columns)}"
    )

axis_anchors_df["axis_label"] = (
    axis_anchors_df["axis_label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

light_anchors = axis_anchors_df.loc[
    axis_anchors_df["axis_label"] == "light",
    "sentence_text",
].tolist()

dark_anchors = axis_anchors_df.loc[
    axis_anchors_df["axis_label"] == "dark",
    "sentence_text",
].tolist()

if not light_anchors:
    raise ValueError("No light anchors found.")

if not dark_anchors:
    raise ValueError("No dark anchors found.")

print(f"Number of light anchors: {len(light_anchors)}")
print(f"Number of dark anchors: {len(dark_anchors)}")

## Encode anchor sentences

In [ ]:
# Cell — Encode semantic-axis anchors

light_embeddings = model.encode(
    light_anchors,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

dark_embeddings = model.encode(
    dark_anchors,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Light embeddings shape:", light_embeddings.shape)
print("Dark embeddings shape:", dark_embeddings.shape)

## Compute Light and Dark centroids

In [ ]:
# Cell — Compute semantic pole centroids

def l2_normalise(vector: np.ndarray) -> np.ndarray:
    """Return an L2-normalised copy of a vector."""
    return vector / np.maximum(np.linalg.norm(vector), 1e-12)


light_centroid = l2_normalise(light_embeddings.mean(axis=0))
dark_centroid = l2_normalise(dark_embeddings.mean(axis=0))

print("Light centroid shape:", light_centroid.shape)
print("Dark centroid shape:", dark_centroid.shape)

The centroid is computed by averaging over the anchor-sentence dimension. This preserves the model's embedding dimension while producing one representative vector for each semantic pole.

## Define the semantic axis

In [ ]:
# Cell — Define Light–Dark semantic axis

axis_midpoint = (light_centroid + dark_centroid) / 2
light_dark_axis = l2_normalise(light_centroid - dark_centroid)

print("Axis shape:", light_dark_axis.shape)

The axis is oriented so that positive values indicate movement toward the Light pole and negative values indicate movement toward the Dark pole.

## Define scoring functions

text → embedding → centre around midpoint → project onto Light–Dark axis → score

In [ ]:
def project_score_from_embedding(
    embedding: np.ndarray,
    midpoint: np.ndarray,
    axis_unit: np.ndarray,
) -> float:
    """
    Project one embedding onto the semantic axis.

    The embedding is first centred relative to the midpoint between the two
    semantic pole centroids. Positive values indicate movement toward the
    Light pole; negative values indicate movement toward the Dark pole.
    """
    centered = embedding - midpoint
    return float(np.dot(centered, axis_unit))


def project_score_from_text(
    text: str,
    model: SentenceTransformer,
    midpoint: np.ndarray,
    axis_unit: np.ndarray,
) -> float:
    """
    Encode one text string and return its semantic-axis score.
    """
    embedding = model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    return project_score_from_embedding(
        embedding=embedding,
        midpoint=midpoint,
        axis_unit=axis_unit,
    )

In [ ]:
def project_scores_from_texts(
    texts: list[str],
    model: SentenceTransformer,
    midpoint: np.ndarray,
    axis_unit: np.ndarray,
    batch_size: int = 64,
) -> np.ndarray:
    """
    Encode multiple texts and project them onto the semantic axis.
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    centered_embeddings = embeddings - midpoint
    return centered_embeddings @ axis_unit

In [ ]:
segments_df["light_dark_score"] = project_scores_from_texts(
    texts=segments_df["segment_text"].tolist(),
    model=model,
    midpoint=axis_midpoint,
    axis_unit=light_dark_axis,
    batch_size=64,
)

In [ ]:
# Cell — Inspect strongest Light and Dark segments

display_cols = [
    "chapter",
    "segment_type",
    "token_count",
    "light_dark_score",
    "segment_text",
]

most_light = (
    segments_df
    .sort_values("light_dark_score", ascending=False)
    [display_cols]
    .head(10)
)

most_dark = (
    segments_df
    .sort_values("light_dark_score", ascending=True)
    [display_cols]
    .head(10)
)

most_light, most_dark

## Sanity-check anchor scores

In [ ]:
light_scores = [
    project_score_from_embedding(e, axis_midpoint, light_dark_axis)
    for e in light_embeddings
]

dark_scores = [
    project_score_from_embedding(e, axis_midpoint, light_dark_axis)
    for e in dark_embeddings
]

anchor_df = pd.DataFrame(
    {
        "set": ["LIGHT"] * len(light_anchors) + ["DARK"] * len(dark_anchors),
        "score": light_scores + dark_scores,
    }
)

anchor_df.sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:
print("Mean Light anchor score:", np.mean(light_scores))
print("Mean Dark anchor score:", np.mean(dark_scores))
print("Min Light anchor score:", np.min(light_scores))
print("Max Dark anchor score:", np.max(dark_scores))

## Check internal coherence of anchor sets

In [ ]:
light_sim = cosine_similarity(light_embeddings)
dark_sim = cosine_similarity(dark_embeddings)

def mean_off_diagonal(sim_matrix: np.ndarray) -> float:
    n = sim_matrix.shape[0]
    return (sim_matrix.sum() - np.trace(sim_matrix)) / (n * (n - 1))

print("Mean Light intra-set similarity:", mean_off_diagonal(light_sim))
print("Mean Dark intra-set similarity:", mean_off_diagonal(dark_sim))
print(
    "Light vs Dark centroid cosine similarity:",
    cosine_similarity([light_centroid], [dark_centroid])[0, 0],
)

## Detect possible anchor outliers

In [ ]:
def centroid_similarity_scores(embeddings: np.ndarray, centroid: np.ndarray) -> np.ndarray:
    return cosine_similarity(embeddings, centroid.reshape(1, -1)).flatten()

light_to_light_centroid = centroid_similarity_scores(
    light_embeddings,
    light_centroid,
)

dark_to_dark_centroid = centroid_similarity_scores(
    dark_embeddings,
    dark_centroid,
)
light_outlier_df = pd.DataFrame(
    {
        "anchor_id": range(len(light_anchors)),
        "axis_label": "light",
        "similarity_to_light_centroid": light_to_light_centroid,
    }
).sort_values("similarity_to_light_centroid")

dark_outlier_df = pd.DataFrame(
    {
        "anchor_id": range(len(dark_anchors)),
        "axis_label": "dark",
        "similarity_to_dark_centroid": dark_to_dark_centroid,
    }
).sort_values("similarity_to_dark_centroid")

print("Possible Light outliers:")
display(light_outlier_df.head())

print("\nPossible Dark outliers:")
display(dark_outlier_df.head())

# Visualisation

In [ ]:
# Cell — Prepare smoothed semantic-axis trajectory

WINDOW = 20

plot_df = segments_df.copy()
plot_df["segment_index"] = range(len(plot_df))
plot_df["smoothed_light_dark_score"] = (
    plot_df["light_dark_score"]
    .rolling(window=WINDOW, center=True, min_periods=1)
    .mean()
)

plot_df.head()

In [ ]:
def plot_semantic_axis_trajectory(
    df: pd.DataFrame,
    score_col: str = "smoothed_light_dark_score",
    chapter_col: str = "chapter",
    title: str = "Light–Dark semantic-axis trajectory",
    figsize: tuple[int, int] = (18, 6),
):
    required_cols = {"segment_index", score_col, chapter_col}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns for plotting: {missing}")

    x = df["segment_index"].to_numpy()
    y = df[score_col].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)

    ax.plot(
        x,
        y,
        color="black",
        linewidth=1.8,
        label=f"{score_col} (window={WINDOW})",
    )

    ax.axhline(0, linewidth=1, linestyle="--")

    ax.fill_between(
        x,
        y,
        0,
        where=y >= 0,
        color="goldenrod",
        alpha=0.25,
        interpolate=True,
        label="Light side",
    )

    ax.fill_between(
        x,
        y,
        0,
        where=y < 0,
        color="midnightblue",
        alpha=0.25,
        interpolate=True,
        label="Dark side",
    )

    chapter_starts = df.groupby(chapter_col)["segment_index"].min()
    chapter_mids = df.groupby(chapter_col)["segment_index"].median()

    for _, start in chapter_starts.items():
        ax.axvline(start, linewidth=0.5, alpha=0.25)

    ax.set_xticks(chapter_mids.values)
    ax.set_xticklabels(chapter_mids.index, rotation=90, fontsize=8)

    ax.set_xlabel("Chapter")
    ax.set_ylabel("Light (+) ↔ Dark (−)")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_semantic_axis_trajectory(
    plot_df,
    score_col="smoothed_light_dark_score",
    chapter_col="chapter",
    title=f"Light–Dark semantic-axis trajectory (rolling window = {WINDOW})",
)